# Import Library

In [1]:
import emoji
import pandas as pd
import re
import ast
import numpy as np

# Import Dataset

In [2]:
df = pd.read_csv("data_with_true_label_fixed.csv", encoding='utf-8-sig')
df

,post_shortcode,post_date,commenter_username,comment_text,comment_likes,true_aspect_1,true_aspect_2,true_sentiment,type
0,DFKccJJPeW_,23/01/2025 09:32,tiara_180319,Pemasangan baru tidak ada kelanjutan setelah s...,0,Pemasangan,Pelayanan,Negatif,Pernyataan
1,DFKccJJPeW_,23/01/2025 09:32,dshellysss,Sama naik semua ini juga komplen..biasanya cum...,0,Harga,Harga,Negatif,Pernyataan
2,DFKccJJPeW_,23/01/2025 09:32,shicya_cute,Min kalau puteran PDAM dol itu sya ngadu kesia...,0,Meteran (Macet/Bermasalah),Meteran (Macet/Bermasalah),Netral,Pertanyaan
3,DFKccJJPeW_,23/01/2025 09:32,iirmawulan,Denda PDAM telat ngalah2in pinjol 50k jadi 150...,1,Harga,Harga,Negatif,Pernyataan
4,DFKccJJPeW_,23/01/2025 09:32,aini_al_aydrus,Air ya klo pagi dan jam 3 sore kenapa selalu mati,1,Air Tidak Mengalir,Air Tidak Mengalir,Negatif,Pertanyaan
...,...,...,...,...,...,...,...,...,...
743,DGe2WlxPbvn,25/02/2025 04:15,mell0dz,gunung anyar tambak apa juga terdampak yaa kok...,1,Air Tidak Mengalir,Air Tidak Mengalir,Netral,Pertanyaan
744,DGe2WlxPbvn,25/02/2025 04:15,dodo_hd12,"Hasilnya gimana, min @pdamsuryasembada ??",1,Lainnya,Lainnya,Netral,Pertanyaan
745,DGcY2FIPTcc,24/02/2025 05:20,adiguna2022,min. daerah sidodadi ada pipa besar bocor. moh...,1,Kebocoran,Kebocoran,Negatif,Pernyataan
746,DGZ7MuDvXk3,23/02/2025 06:22,nikelu22,Min. Ini pdam saya mati sekitar tgl segini. Sa...,0,Air Tidak Mengalir,Pelayanan,Netral,Pertanyaan


# Removing Duplicates

In [3]:
df.duplicated().sum()

np.int64(2)

In [4]:
df = df.drop_duplicates()
df = df.reset_index(drop=True)
df.duplicated().sum()

np.int64(0)

# Preparing Emoji Dataset

In [5]:
df_emoji = pd.read_csv('emoji.csv', index_col=0)

In [6]:
# Ganti spasi dengan underscore pada kolom 'name'
df_emoji['name'] = df_emoji['name'].str.replace(' ', '_')
df_emoji['name'] = df_emoji['name'].str.replace('-', '_')
df_emoji['name'] = df_emoji['name'].str.replace('⊛_', '')
df_emoji['name'] = df_emoji['name'].str.replace('"', '')
df_emoji['name'] = df_emoji['name'].str.replace(':', '')
df_emoji['name'] = df_emoji['name'].str.replace(',', '')
df_emoji['name'] = df_emoji['name'].str.replace("'", "")
# Tampilkan 5 baris pertama untuk memastikan hasilnya
print(df_emoji.head())

                              name
0                    grinning_face
1      grinning_face_with_big_eyes
2  grinning_face_with_smiling_eyes
3   beaming_face_with_smiling_eyes
4          grinning_squinting_face


In [7]:
df_emoji.to_csv('emoji_underscore.csv', index=True)

In [8]:
df_emoji

,name
0,grinning_face
1,grinning_face_with_big_eyes
2,grinning_face_with_smiling_eyes
3,beaming_face_with_smiling_eyes
4,grinning_squinting_face
...,...
1811,flag_Zambia
1812,flag_Zimbabwe
1813,flag_England
1814,flag_Scotland


# Cleansing

In [9]:
# --- 1. Baca daftar emoji dari file emoji_underscore.csv ---
emoji_df = pd.read_csv('emoji_underscore.csv')
emoji_df['name'] = emoji_df['name'].astype(str).str.strip()

# Buat pola regex berdasarkan nama emoji (format :emoji_name:)
emoji_names = [re.escape(name) for name in emoji_df['name'] if name]
emoji_pattern = re.compile(r':(' + '|'.join(emoji_names) + r'):', flags=re.IGNORECASE)


# --- 2. Fungsi pembersihan teks ---
def clean_text(text):
    if pd.isna(text):
        return ''
    
    text = str(text).lower()

    # Hapus emoji berdasarkan daftar emoji
    text = emoji_pattern.sub('', text)

    # Hapus URL
    text = re.sub(r'http\S+|www.\S+', '', text)
    
    # Hapus tanda hubung "-"
    text = re.sub('-', ' ', text)
    
    # Hapus mention dan hashtag
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # Hapus angka dan simbol non-huruf
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


# Terapkan fungsi clean_text ke kolom comment_text
df['cleaned_text'] = df['comment_text'].apply(clean_text)

# --- 4. Hapus baris yang kosong atau hanya berisi emoji (setelah dibersihkan jadi kosong) ---
before = len(df)
df = df[df['cleaned_text'].str.strip().astype(bool)]
after = len(df)

print(f"🧹 {before - after} baris dihapus karena hanya berisi emoji atau kosong.")

🧹 55 baris dihapus karena hanya berisi emoji atau kosong.


In [10]:
print(df['cleaned_text'].to_markdown())

|     | cleaned_text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|----:|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Tokenize + Stopword Removal + Formalization

In [ ]:
import nltk
import string
from nltk.tokenize import word_tokenize

# Pastikan tokenizer sudah diunduh
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# --- Fungsi Slang Removal + Stopword Filtering + Tokenization ---
def slang_remove_tokenize(text):
    if pd.isna(text) or not isinstance(text, str):
        return []

    # Baca daftar stopwords
    with open("combined_stop_words.txt", "r", encoding="utf-8") as f:
        stop_words = f.read().splitlines()

    # Baca daftar slang words
    with open("update_combined_slang_words.txt", "r", encoding="utf-8") as f:
        slang_words = ast.literal_eval(f.read())

    # Tokenisasi teks
    word_tokens = word_tokenize(text.lower())

    # Ganti slang dengan padanan katanya
    normalized_tokens = [slang_words.get(w, w) for w in word_tokens]

    # Filter stopword dan tanda baca
    filtered_tokens = [
        w for w in normalized_tokens 
        if w not in stop_words and w not in string.punctuation and w.strip()
    ]

    return filtered_tokens  # bisa juga ' '.join(filtered_tokens) jika ingin string

df['tokens'] = df['cleaned_text'].apply(slang_remove_tokenize)

print(df[['cleaned_text', 'tokens']].to_markdown())

|     | cleaned_text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            | tokens                                                                                                                                                                                                                

# Drop Empty row

In [12]:
# --- 2️⃣ Jika kolom tokens disimpan dalam bentuk string list (contoh: "['air', 'keruh']") ---
# Ubah string list menjadi list Python sesungguhnya
def safe_eval(x):
    try:
        if isinstance(x, str) and x.startswith('[') and x.endswith(']'):
            return ast.literal_eval(x)
        elif isinstance(x, float) and pd.isna(x):
            return []
        else:
            return x
    except:
        return []

df['tokens'] = df['tokens'].apply(safe_eval)

# --- 3️⃣ Hapus baris dengan kolom tokens kosong ---
before = len(df)
df = df[df['tokens'].apply(lambda x: isinstance(x, list) and len(x) > 0)]
after = len(df)

print(f"✅ Baris kosong pada kolom 'tokens' berhasil dihapus.")
print(f"🗑️ Total baris dihapus: {before - after}")
# df.to_csv("cleaned_tokens_dataset.csv", sep = ';', index=False, encoding='utf-8-sig')


✅ Baris kosong pada kolom 'tokens' berhasil dihapus.
🗑️ Total baris dihapus: 12


# Stemming

In [13]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()
df['final_data'] = df['tokens'].apply(lambda x: stemmer.stem(" ".join(x)) if isinstance(x, list) else stemmer.stem(x))
print(df['final_data'].to_markdown())
df[['post_shortcode', 'commenter_username', 'comment_text', 'cleaned_text', 'tokens', 'final_data', 'true_aspect_1', 'true_aspect_2', 'true_sentiment', 'type']].to_csv("final_stemmed_dataset.csv", index=False, encoding='utf-8-sig')

|     | final_data                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
|----:|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Split Data Train: 100% Test: 100%

In [14]:
df_test = df
df_train = df

## Feature Extraction & Aspect Based Classification

In [15]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# 1️⃣ VECTORISASI TF-IDF
# =========================================================

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2
)

X_train = vectorizer.fit_transform(df_train['final_data'])
X_test  = vectorizer.transform(df_test['final_data'])

# =========================================================
# 2️⃣ COSINE SIMILARITY
# =========================================================

cosine_sim = cosine_similarity(X_test, X_train)

THRESHOLD = 0.3
TOP_K = 2

# =========================================================
# 3️⃣ TANDAI DATA MULTI-ASPECT SEBENARNYA
# =========================================================

def is_multi_aspect(row):
    return (
        pd.notna(row['true_aspect_1']) and
        pd.notna(row['true_aspect_2']) and
        row['true_aspect_1'] != 'Lainnya' and
        row['true_aspect_2'] != 'Lainnya' and
        row['true_aspect_1'] != row['true_aspect_2']
    )

df_test['is_multi_aspect'] = df_test.apply(is_multi_aspect, axis=1)

# =========================================================
# 4️⃣ PRE-COMPUTE INDEX PER ASPEK
# =========================================================

unique_aspects = df_train['true_aspect_1'].unique()
train_aspect_array = df_train['true_aspect_1'].to_numpy()

aspect_to_positions = {
    aspect: np.where(train_aspect_array == aspect)[0]
    for aspect in unique_aspects
}

# =========================================================
# 5️⃣ MULTI-ASPECT CLASSIFICATION
# =========================================================

predicted_aspects = []
similarity_scores = []

for i in range(cosine_sim.shape[0]):
    row = cosine_sim[i]

    # ----------------------------------
    # SIMILARITY MAKSIMAL PER ASPEK
    # ----------------------------------
    aspect_sim_dict = {
        aspect: row[pos].max() if len(pos) > 0 else 0.0
        for aspect, pos in aspect_to_positions.items()
    }

    # ----------------------------------
    # TOP-2 GLOBAL SIMILARITY
    # ----------------------------------
    sorted_idx = row.argsort()[::-1]
    idx1, idx2 = sorted_idx[:2]

    score1, score2 = row[idx1], row[idx2]
    aspect1 = df_train.iloc[idx1]['true_aspect_1']
    aspect2 = df_train.iloc[idx2]['true_aspect_1']

    # ----------------------------------
    # SINGLE-ASPECT
    # ----------------------------------
    if not df_test.iloc[i]['is_multi_aspect']:
        predicted_aspects.append(aspect1 if score1 >= THRESHOLD else "Lainnya")
        similarity_scores.append(score1)

    # ----------------------------------
    # MULTI-ASPECT
    # ----------------------------------
    else:
        aspects, scores = [], []

        if score1 >= THRESHOLD:
            aspects.append(aspect1)
            scores.append(score1)

        if score2 >= THRESHOLD and aspect2 != aspect1:
            aspects.append(aspect2)
            scores.append(score2)

        predicted_aspects.append(
            " | ".join(sorted(aspects)) if aspects else "Lainnya"
        )
        similarity_scores.append(max(scores) if scores else score1)

    # =====================================================
    # 🔎 DEBUG PRINT (SEMUA ASPEK)
    # =====================================================
    print("=" * 80)
    print(f"Index Test   : {i}")
    print(f"Teks         : {df_test.iloc[i]['final_data']}")
    print(f"True Aspect  : {df_test.iloc[i]['true_aspect_1']} | {df_test.iloc[i]['true_aspect_2']}")
    print(f"Predicted    : {predicted_aspects[-1]}")
    print("Similarity tiap aspek:")
    for asp, sim in aspect_sim_dict.items():
        print(f"  - {asp:25s}: {sim:.4f}")
    print("=" * 80 + "\n")

# =========================================================
# 6️⃣ SIMPAN HASIL (HANYA KOLOM YANG DIMINTA)
# =========================================================

df_test['predicted_aspects'] = predicted_aspects
df_test['similarity_scores'] = similarity_scores

output_cols = [
    'post_shortcode',
    'post_date',
    'commenter_username',
    'comment_text',
    'comment_likes',
    'true_aspect_1',
    'true_aspect_2',
    'true_sentiment',
    'type',
    'cleaned_text',
    'tokens',
    'final_data',
    'predicted_aspects',
    'similarity_scores'
]

df_test[output_cols].to_csv(
    "trial_multi_aspect_classification_result.csv",
    index=False,
    encoding='utf-8-sig'
)
print("✅ Multi-aspect classification selesai")

Index Test   : 0
Teks         : pasang tidak lanjut survey kata komplain layan langgan whatsapp sedang whatsapp tidak tanggap
True Aspect  : Pemasangan | Pelayanan
Predicted    : Pelayanan | Pemasangan
Similarity tiap aspek:
  - Pemasangan               : 1.0000
  - Harga                    : 0.1850
  - Meteran (Macet/Bermasalah): 0.1999
  - Air Tidak Mengalir       : 0.3218
  - Pelayanan                : 0.3700
  - Air Kotor/Bau            : 0.0977
  - Lainnya                  : 0.1710
  - Kebocoran                : 0.0264

Index Test   : 1
Teks         : naik komplain biasa an aduh serba bayar kan ga percaya perintah
True Aspect  : Harga | Harga
Predicted    : Harga
Similarity tiap aspek:
  - Pemasangan               : 0.0830
  - Harga                    : 1.0000
  - Meteran (Macet/Bermasalah): 0.1202
  - Air Tidak Mengalir       : 0.3232
  - Pelayanan                : 0.1023
  - Air Kotor/Bau            : 0.1502
  - Lainnya                  : 0.2800
  - Kebocoran                : 0.

# Aspect Based Evaluation

In [16]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# =========================================================
# 1️⃣ BENTUK SET TRUE ASPECT (MULTI)
# =========================================================

def build_true_aspect_set(row):
    aspects = set()

    if pd.notna(row['true_aspect_1']) and row['true_aspect_1'] != 'Lainnya':
        aspects.add(row['true_aspect_1'])

    if 'true_aspect_2' in row and pd.notna(row['true_aspect_2']) and row['true_aspect_2'] != 'Lainnya':
        aspects.add(row['true_aspect_2'])

    if len(aspects) == 0:
        aspects.add('Lainnya')

    return aspects

df_test['true_aspects_set'] = df_test.apply(build_true_aspect_set, axis=1)

# =========================================================
# 2️⃣ BENTUK SET PREDICTED ASPECT (MULTI)
# =========================================================

def build_predicted_aspect_set(pred):
    if pred == 'Lainnya':
        return {'Lainnya'}
    return set([a.strip() for a in pred.split('|')])

df_test['predicted_aspects_set'] = df_test['predicted_aspects'].apply(build_predicted_aspect_set)

print(f"Total Data Train : {len(df_train)}")
print(f"Total Data Test  : {len(df_test)}")

# =========================================================
# 3️⃣ EVALUASI MULTI-ASPECT
# =========================================================

# Exact Match
df_test['exact_match'] = (
    df_test['true_aspects_set'] == df_test['predicted_aspects_set']
)

exact_accuracy = df_test['exact_match'].mean()

# Partial Match
df_test['partial_match'] = df_test.apply(
    lambda x: len(x['true_aspects_set'].intersection(x['predicted_aspects_set'])) > 0,
    axis=1
)

partial_accuracy = df_test['partial_match'].mean()

print("\n==============================")
print("🎯 EVALUASI MULTI-ASPECT")
print("==============================")
print(f"Exact Match Accuracy   : {exact_accuracy:.4f}")
print(f"Partial Match Accuracy : {partial_accuracy:.4f}")

# =========================================================
# 4️⃣ STATISTIK PREDICTED ASPECT
# =========================================================

aspect_stats = (
    df_test
    .groupby('predicted_aspects')
    .agg(
        Jumlah_Kemunculan=('predicted_aspects', 'count'),
        Rata_rata_Similarity=('similarity_scores', 'mean')
    )
    .reset_index()
    .sort_values('Jumlah_Kemunculan', ascending=False)
)

print("\n===========================================")
print("📊 Statistik Predicted Aspect (Test Data)")
print("===========================================")
print(aspect_stats.to_markdown(index=False))

# =========================================================
# 5️⃣ DISTRIBUSI TRUE ASPECT (MULTI)
# =========================================================

true_aspect_exploded = (
    df_test['true_aspects_set']
    .apply(list)
    .explode()
    .value_counts()
    .reset_index()
)

true_aspect_exploded.columns = ['Aspek', 'Jumlah']

print("\n==================================================")
print("📌 Distribusi True Aspect (Multi-Aspect)")
print("==================================================")
print(true_aspect_exploded.to_markdown(index=False))

# =========================================================
# 6️⃣ TOTAL SALAH & BENAR (BERDASARKAN EXACT MATCH)
# =========================================================

total_correct = df_test['exact_match'].sum()
total_wrong   = len(df_test) - total_correct

print("\n===========================================")
print("❌ HASIL KLASIFIKASI MULTI-ASPECT")
print("===========================================")
print(f"Total Data Test      : {len(df_test)}")
print(f"Prediksi Benar (Exact): {total_correct}")
print(f"Prediksi Salah       : {total_wrong}")
print(f"Error Rate           : {total_wrong/len(df_test)*100:.2f}%")

# =========================================================
# 7️⃣ DATA SALAH PREDIKSI (EXACT MATCH)
# =========================================================

misclassified_df = df_test.loc[
    ~df_test['exact_match'],
    ['final_data', 'true_aspects_set', 'predicted_aspects_set', 'similarity_scores']
]

misclassified_df.to_csv(
    "trial_misclassified_multi_aspect_data.csv",
    index=False,
    encoding='utf-8-sig'
)

Total Data Train : 679
Total Data Test  : 679

🎯 EVALUASI MULTI-ASPECT
Exact Match Accuracy   : 0.9323
Partial Match Accuracy : 1.0000

📊 Statistik Predicted Aspect (Test Data)
| predicted_aspects                       |   Jumlah_Kemunculan |   Rata_rata_Similarity |
|:----------------------------------------|--------------------:|-----------------------:|
| Lainnya                                 |                 236 |               0.949153 |
| Air Tidak Mengalir                      |                 215 |               1        |
| Pelayanan                               |                  50 |               1        |
| Harga                                   |                  43 |               1        |
| Air Kotor/Bau                           |                  39 |               1        |
| Meteran (Macet/Bermasalah)              |                  36 |               1        |
| Pemasangan                              |                  14 |               1        |
| Ai